<a href="https://colab.research.google.com/github/austinmallie/ADS599_Capstone/blob/main/Capstone_Streamplit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Introduction & Acknowledgements

## App Overview — Hospital Risk Explorer

### Purpose
The Hospital Risk Explorer is an interactive decision-support tool designed to help
analysts and policymakers identify and prioritize hospitals at elevated risk of
ransomware incidents. Rather than presenting static model outputs, the app creates
a dynamic workflow where the user controls the analytical assumptions and sees the
consequences of those decisions in real time.

---

### The Core Question
**"Which hospitals should we prioritize, and at what threshold?"**

Machine learning models produce probability scores, not decisions. This app bridges
that gap — translating raw model output into an actionable prioritization list by
allowing the user to define what level of risk warrants intervention.

---

### How It Works

The app loads a prepared dataset of hospital records, each scored by four trained
machine learning models:

| Model | Description |
|---|---|
| Logistic Regression | Interpretable linear baseline |
| Random Forest | Ensemble tree-based model |
| XGBoost | Gradient boosted trees (default) |
| SVM | Support vector machine |

The user selects a model and sets a **risk threshold** — any hospital whose predicted
probability meets or exceeds that threshold is flagged for review. All metrics,
visualizations, and the flagged hospital table update instantly as controls change.

---

### Left Sidebar Controls

| Control | What It Does |
|---|---|
| **Model selector** | Switches which model's probability scores drive the analysis |
| **Risk threshold slider** | Sets the cutoff probability for flagging a hospital |
| **State filter** | Narrows the analysis to one or more states |
| **Hospital type filter** | Filters by facility type (Acute Care, Rural/Access, etc.) |

---

### Main Page Output

**Consequence Statement**
A plain-English summary at the top of the page translates the current threshold
and model selection into real numbers — how many hospitals are flagged, how many
truly high-risk hospitals are caught, how many are missed, and how many flags are
false alarms. The banner color updates from red to orange to green as recall improves.

**KPI Row**
Six at-a-glance metrics give an immediate read on model performance under the
current settings:
- 🚩 Flagged — total hospitals flagged for review
- ✅ Caught (TP) — true positives, truly high-risk hospitals correctly identified
- ❌ Missed (FN) — false negatives, high-risk hospitals the model failed to flag
- ⚠️ False Alarms (FP) — low-risk hospitals incorrectly flagged
- 📡 Recall — share of truly high-risk hospitals captured
- 🎯 Precision — share of flagged hospitals that are genuinely high-risk

**Flagged Hospital Table**
A sortable, color-coded table of every flagged hospital showing:
- Risk tier (Critical / High / Medium) based on how far above the threshold the score falls
- All four model probability scores for comparison
- Key hospital context including state, type, ownership, and bed tier
- Top model features including current ratio, FTE employees, engagement index,
  inpatient and outpatient revenue share
- Ground truth label (Actual High-Risk) for validation

**Download Button**
Exports the current flagged hospital table as a CSV, with the model name and
threshold embedded in the filename for traceability.

---

### Why This Matters
Static model outputs require a data scientist to interpret them. This app puts
analytical control directly in the hands of the decision-maker — allowing a
hospital administrator, regulator, or policy analyst to explore tradeoffs between
catching more high-risk hospitals (higher recall) versus generating fewer false
alarms (higher precision) without writing a single line of code.

## AI Acknowledgements

This Streamlit application interface was developed with the assistance of Claude (Anthropic, 2025), a large language model used as an interactive coding tutor. The author directed all design decisions, including layout specifications, metric selection, risk tier definitions, threshold logic, and feature display choices. Claude provided Streamlit syntax scaffolding, indentation corrections, and debugging support. All analytical methodology, model development, feature engineering, and research conclusions are the author's original work.

Anthropic. (2025). Claude (claude-sonnet-4-20250514) [Large language model]. https://www.anthropic.com

#Setup

In [1]:
#install ngrok for hosting
!pip install streamlit pyngrok -q

## Model Results Data

In [2]:
#connect to data, results
import pandas as pd
url = "https://raw.githubusercontent.com/austinmallie/ADS599_Capstone/refs/heads/main/Data-Folder/app_data.csv"
df_results = pd.read_csv(url)
df_results.head()

,ccn,report_year,y_true,LogisticRegression_proba,LogisticRegression_pred,RandomForest_proba,RandomForest_pred,XGBoost_proba,XGBoost_pred,SV_proba,SVM_pred
0,10001,2021,0,0.484856,0,0.658950,1,0.581531,1,0.026622,0
1,10005,2021,0,0.842459,1,0.542228,1,0.153712,0,0.579800,1
2,10006,2021,0,0.426835,0,0.161748,0,0.060532,0,0.001786,0
3,10007,2021,0,0.754120,1,0.209973,0,0.031119,0,0.000832,0
4,10008,2021,0,0.319911,0,0.159571,0,0.038359,0,0.082715,1


In [3]:
df_results.columns

Index(['ccn', 'report_year', 'y_true', 'LogisticRegression_proba',
       'LogisticRegression_pred', 'RandomForest_proba', 'RandomForest_pred',
       'XGBoost_proba', 'XGBoost_pred', 'SV_proba', 'SVM_pred'],
      dtype='object')

## Hospital General Data

In [4]:
#connect to data, hospital general
url = "https://raw.githubusercontent.com/austinmallie/ADS599_Capstone/refs/heads/main/Data-Folder/Hospital-Geneneral-Data/Hospital_General_Clean_Final.csv"
df_hospital = pd.read_csv(url)
df_hospital.head()

,Facility ID,State,Hospital_Type,Birthing_Center,Emergency_Services,Hospital_Ownership,Hospital_Rating,MORT_Participation,Safety_Participation,READM_Participation,...,Domains_Reporting,MORT_Better,MORT_Worse,Safety_Better,Safety_Worse,READM_Better,READM_Worse,Net_Performance_Score,Net_Performance_Rate,Performance_Ratio
0,010001,AL,Acute Care,1,1,Government,4.0,1.000000,0.875,1.000000,...,5,0.0,0.000000,0.428571,0.0,0.0,0.000000,3.0,0.120000,0.120
1,010005,AL,Acute Care,1,1,Government,3.0,0.857143,0.875,0.818182,...,5,0.0,0.166667,0.000000,0.0,0.0,0.111111,-2.0,-0.090909,0.000
2,010006,AL,Acute Care,1,1,For-Profit,2.0,1.000000,1.000,0.818182,...,5,0.0,0.285714,0.375000,0.0,0.0,0.111111,0.0,0.000000,0.125
3,010007,AL,Acute Care,0,1,Non-Profit,1.0,0.428571,0.375,0.636364,...,5,0.0,0.333333,0.000000,0.0,0.0,0.000000,-1.0,-0.076923,0.000
4,010008,AL,Acute Care,0,1,For-Profit,NaN,0.142857,0.000,0.181818,...,3,0.0,0.000000,NaN,NaN,0.0,0.000000,0.0,0.000000,0.000


In [5]:
#rename facility ID to ccn
df_hospital = df_hospital.rename(columns={'Facility ID': 'ccn'})

In [6]:
#investigat hospital general columns
df_hospital.columns

Index(['ccn', 'State', 'Hospital_Type', 'Birthing_Center',
       'Emergency_Services', 'Hospital_Ownership', 'Hospital_Rating',
       'MORT_Participation', 'Safety_Participation', 'READM_Participation',
       'PtExp_Participation', 'TE_Participation', 'Engagement_Index',
       'Domains_Reporting', 'MORT_Better', 'MORT_Worse', 'Safety_Better',
       'Safety_Worse', 'READM_Better', 'READM_Worse', 'Net_Performance_Score',
       'Net_Performance_Rate', 'Performance_Ratio'],
      dtype='object')

In [7]:
# Drop non-numeric CCNs and clean
df_hospital = df_hospital[
    df_hospital["ccn"].astype(str).str.strip().str.lstrip("0").str.match(r'^\d+$')
].copy()

df_hospital["ccn"] = df_hospital["ccn"].astype(str).str.strip().str.lstrip("0").astype(int)

print(f"df_hospital rows after filtering: {len(df_hospital):,}")
df_hospital.head()

df_hospital rows after filtering: 5,262


,ccn,State,Hospital_Type,Birthing_Center,Emergency_Services,Hospital_Ownership,Hospital_Rating,MORT_Participation,Safety_Participation,READM_Participation,...,Domains_Reporting,MORT_Better,MORT_Worse,Safety_Better,Safety_Worse,READM_Better,READM_Worse,Net_Performance_Score,Net_Performance_Rate,Performance_Ratio
0,10001,AL,Acute Care,1,1,Government,4.0,1.000000,0.875,1.000000,...,5,0.0,0.000000,0.428571,0.0,0.0,0.000000,3.0,0.120000,0.120
1,10005,AL,Acute Care,1,1,Government,3.0,0.857143,0.875,0.818182,...,5,0.0,0.166667,0.000000,0.0,0.0,0.111111,-2.0,-0.090909,0.000
2,10006,AL,Acute Care,1,1,For-Profit,2.0,1.000000,1.000,0.818182,...,5,0.0,0.285714,0.375000,0.0,0.0,0.111111,0.0,0.000000,0.125
3,10007,AL,Acute Care,0,1,Non-Profit,1.0,0.428571,0.375,0.636364,...,5,0.0,0.333333,0.000000,0.0,0.0,0.000000,-1.0,-0.076923,0.000
4,10008,AL,Acute Care,0,1,For-Profit,NaN,0.142857,0.000,0.181818,...,3,0.0,0.000000,NaN,NaN,0.0,0.000000,0.0,0.000000,0.000


## Dataset Modeling on

In [8]:
#connect to data, modeling
from google.colab import userdata
import os

token    = userdata.get('Github')
owner    = "austinmallie"
repo     = "ADS599_Capstone"
repo_url = f"https://{token}@github.com/{owner}/{repo}.git"

if not os.path.exists(repo):
    !git clone {repo_url}
else:
    print("Repo already cloned. Pulling latest changes...")
    %cd {repo}
    !git pull

%cd /content/{repo}

Repo already cloned. Pulling latest changes...
/content/ADS599_Capstone
Already up to date.
/content/ADS599_Capstone


In [9]:
#get dataset
file_path = "/content/ADS599_Capstone/Models/hospital_ransomware_model.parquet"
df_model = pd.read_parquet(file_path)
df_model.head()


,ccn,cbsa,fte_employees,beds,total_discharges,total_unreimbursed_and_uncompensated_care,total_costs,cash_on_hand_and_in_banks,total_assets,total_liabilities,...,mort_worse,safety_better,safety_worse,readm_better,readm_worse,net_performance_score,net_performance_rate,performance_ratio,total_individuals_affected,had_ransomware
0,10001,20020.0,2405.80,398.0,18489.0,28691031.0,231992999.0,5836798.0,393203543.0,155249153.0,...,0.0,0.428571,0.0,0.0,0.0,3.0,0.12,0.12,0.0,0
1,10001,20020.0,2379.50,400.0,18484.0,25026782.0,238867113.0,8339540.0,391583329.0,152831940.0,...,0.0,0.428571,0.0,0.0,0.0,3.0,0.12,0.12,0.0,0
2,10001,20020.0,2397.43,387.0,19632.0,29216156.0,246974731.0,6085744.0,383657576.0,142863228.0,...,0.0,0.428571,0.0,0.0,0.0,3.0,0.12,0.12,0.0,0
3,10001,20020.0,2425.37,387.0,19677.0,21713434.0,250424328.0,31651152.0,391259641.0,140218773.0,...,0.0,0.428571,0.0,0.0,0.0,3.0,0.12,0.12,0.0,0
4,10001,20020.0,2371.67,327.0,19963.0,21782404.0,262391792.0,32282079.0,390435696.0,146622307.0,...,0.0,0.428571,0.0,0.0,0.0,3.0,0.12,0.12,0.0,0


In [10]:
#drop everything that is not 2021
df_model = df_model[df_model['report_year'] == 2021]
df_model.shape

(6053, 64)

In [11]:
# Step 1: Check the original source
print("In df_model:", 'fte_employees' in df_model.columns)
print(f"df_model shape: {df_model.shape}")

In df_model: True
df_model shape: (6053, 64)


In [12]:
#resolve duplicate CCNs but unique rows found based on a later step

#find the dupes
dupe_mask  = df_model.duplicated(subset=["ccn", "report_year"], keep=False)
df_clean   = df_model[~dupe_mask].copy()
df_dupes   = df_model[dupe_mask].copy()

#find the non numerics
non_numeric_cols = ["ccn", "report_year"] + \
                   df_dupes.select_dtypes(exclude="number").columns.tolist()
numeric_cols     = [c for c in df_dupes.columns if c not in non_numeric_cols]

print(f"Numeric columns to average : {len(numeric_cols)}")
print(f"Non-numeric columns (first): {len(non_numeric_cols)}")

# average the dupes
agg_dict = {col: "mean" for col in numeric_cols}
agg_dict.update({col: "first" for col in non_numeric_cols
                 if col not in ["ccn", "report_year"]})

df_dupes_resolved = (
    df_dupes
    .groupby(["ccn", "report_year"], as_index=False)
    .agg(agg_dict)
)
#recombine
df_model_clean = pd.concat([df_clean, df_dupes_resolved], ignore_index=True)

#validate
remaining_dupes = df_model_clean.duplicated(subset=["ccn", "report_year"]).sum()

print(f"\nOriginal df_model rows  : {len(df_model):,}")
print(f"Clean rows retained     : {len(df_clean):,}")
print(f"Duplicate pairs resolved: {len(df_dupes_resolved):,}")
print(f"Final df_model rows     : {len(df_model_clean):,}")
print(f"Remaining duplicates    : {remaining_dupes}")

Numeric columns to average : 58
Non-numeric columns (first): 6

Original df_model rows  : 6,053
Clean rows retained     : 5,907
Duplicate pairs resolved: 73
Final df_model rows     : 5,980
Remaining duplicates    : 0


In [13]:
df_model.columns

Index(['ccn', 'cbsa', 'fte_employees', 'beds', 'total_discharges',
       'total_unreimbursed_and_uncompensated_care', 'total_costs',
       'cash_on_hand_and_in_banks', 'total_assets', 'total_liabilities',
       'net_income', 'cost_to_charge_ratio', 'medicaid_net_revenue',
       'report_year', 'net_profit_margin', 'operating_margin',
       'cost_to_revenue_ratio', 'outpatient_revenue_share',
       'inpatient_revenue_share', 'salary_cost_share', 'overhead_cost_share',
       'depreciation_cost_share', 'contract_labor_share',
       'charity_care_pct_revenue', 'bad_debt_pct_revenue', 'occupancy_rate',
       'discharge_rate', 'debt_to_equity', 'asset_turnover', 'liability_ratio',
       'current_ratio', 'medicaid_revenue_share',
       'net_patient_revenue_yoy_change', 'total_costs_yoy_change',
       'net_income_yoy_change', 'beds_yoy_change', 'fte_employees_yoy_change',
       'is_profitable', 'high_charity', 'bed_tier', 'revenue_tier',
       'hospital_type', 'birthing_center', '

## Join Datasets

In [14]:
#get state data
df_hospital_geo = df_hospital[["ccn", "State"]].copy()
# merge results + model
merged = df_results.merge(
    df_model_clean,
    on=["ccn", "report_year"],
    how="left"
)
#add state
merged = merged.merge(
    df_hospital_geo,
    on="ccn",
    how="left"
)
merged.head()

,ccn,report_year,y_true,LogisticRegression_proba,LogisticRegression_pred,RandomForest_proba,RandomForest_pred,XGBoost_proba,XGBoost_pred,SV_proba,...,safety_better,safety_worse,readm_better,readm_worse,net_performance_score,net_performance_rate,performance_ratio,total_individuals_affected,had_ransomware,State
0,10001,2021,0,0.484856,0,0.658950,1,0.581531,1,0.026622,...,0.428571,0.0,0.0,0.000000,3.0,0.120000,0.120,0.0,0.0,AL
1,10005,2021,0,0.842459,1,0.542228,1,0.153712,0,0.579800,...,0.000000,0.0,0.0,0.111111,-2.0,-0.090909,0.000,0.0,0.0,AL
2,10006,2021,0,0.426835,0,0.161748,0,0.060532,0,0.001786,...,0.375000,0.0,0.0,0.111111,0.0,0.000000,0.125,0.0,0.0,AL
3,10007,2021,0,0.754120,1,0.209973,0,0.031119,0,0.000832,...,0.000000,0.0,0.0,0.000000,-1.0,-0.076923,0.000,0.0,0.0,AL
4,10008,2021,0,0.319911,0,0.159571,0,0.038359,0,0.082715,...,NaN,NaN,0.0,0.000000,0.0,0.000000,0.000,0.0,0.0,AL


In [15]:
#drop anythign with a missing state
df_app = merged.dropna(subset=["State"])
print(f"Rows after dropping missing State: {len(df_app):,}")
print(f"Rows dropped: {len(merged) - len(df_app):,}")
print(f"Positive rate before: {merged['y_true'].mean():.3f}")
print(f"Positive rate after:  {df_app['y_true'].mean():.3f}")

Rows after dropping missing State: 5,022
Rows dropped: 912
Positive rate before: 0.031
Positive rate after:  0.036


In [16]:
#validate
total     = len(df_results)
matched   = merged["beds"].notna().sum()
geo_match = merged["State"].notna().sum()
unmatched = total - matched

print(f"Prediction rows  : {total:,}")
print(f"Matched (model)  : {matched:,} ({matched/total*100:.1f}%)")
print(f"Matched (geo)    : {geo_match:,} ({geo_match/total*100:.1f}%)")
print(f"Unmatched        : {unmatched:,}")

if unmatched > 0:
    print("\nSample unmatched CCNs:")
    print(merged.loc[merged["beds"].isna(), "ccn"].head(10).tolist())

Prediction rows  : 5,934
Matched (model)  : 5,898 (99.4%)
Matched (geo)    : 5,022 (84.6%)
Unmatched        : 36

Sample unmatched CCNs:
[14000, 14006, 14014, 44021, 50697, 51990, 53306, 53308, 70039, 70040]


In [17]:
df_app.columns

Index(['ccn', 'report_year', 'y_true', 'LogisticRegression_proba',
       'LogisticRegression_pred', 'RandomForest_proba', 'RandomForest_pred',
       'XGBoost_proba', 'XGBoost_pred', 'SV_proba', 'SVM_pred', 'cbsa',
       'fte_employees', 'beds', 'total_discharges',
       'total_unreimbursed_and_uncompensated_care', 'total_costs',
       'cash_on_hand_and_in_banks', 'total_assets', 'total_liabilities',
       'net_income', 'cost_to_charge_ratio', 'medicaid_net_revenue',
       'net_profit_margin', 'operating_margin', 'cost_to_revenue_ratio',
       'outpatient_revenue_share', 'inpatient_revenue_share',
       'salary_cost_share', 'overhead_cost_share', 'depreciation_cost_share',
       'contract_labor_share', 'charity_care_pct_revenue',
       'bad_debt_pct_revenue', 'occupancy_rate', 'discharge_rate',
       'debt_to_equity', 'asset_turnover', 'liability_ratio', 'current_ratio',
       'medicaid_revenue_share', 'net_patient_revenue_yoy_change',
       'total_costs_yoy_change', 'net_

## Investigate Top Features

In [18]:
#look at most important features
url = "https://raw.githubusercontent.com/austinmallie/ADS599_Capstone/main/Models/Feature%20Importance/feature_importance_final.csv"
df_feature_importance = pd.read_csv(url)
df_feature_importance = df_feature_importance.sort_values('Abs_Impact', ascending=False)
top_n = df_feature_importance.head(10)
print(top_n[['Feature', 'Coefficient', 'Abs_Impact']])

                    Feature  Coefficient  Abs_Impact
0   inpatient_revenue_share    -3.409281    3.409281
1  fte_employees_yoy_change    -3.143836    3.143836
2  outpatient_revenue_share    -2.870688    2.870688
3             fte_employees     2.576534    2.576534
4             current_ratio    -1.386993    1.386993
5      cost_to_charge_ratio    -1.348399    1.348399
6         performance_ratio    -1.334855    1.334855
7          engagement_index    -1.304119    1.304119
8      safety_participation     1.151399    1.151399
9      net_performance_rate     1.013579    1.013579


In [19]:
df_feature_importance['Abs_Impact'].describe()

,Abs_Impact
count,68.000000
mean,0.476463
std,0.735271
min,0.001580
25%,0.053131
50%,0.219412
75%,0.465534
max,3.409281


In [20]:
#create a list of low impact to automatically drop
threshold = df_feature_importance['Abs_Impact'].mean() + df_feature_importance['Abs_Impact'].std()
print(f"Threshold: {threshold:.4f}")

low_impact_features = df_feature_importance[df_feature_importance['Abs_Impact'] < threshold]['Feature'].tolist()
print(f"Features to drop: {len(low_impact_features)}")
print(low_impact_features)

Threshold: 1.2117
Features to drop: 60
['safety_participation', 'net_performance_rate', 'readm_worse', 'total_unreimbursed_and_uncompensated_care', 'asset_turnover', 'net_performance_score', 'readm_participation', 'mort_worse', 'discharge_rate', 'total_discharges', 'safety_worse', 'safety_better', 'hospital_type_Psychiatric', 'occupancy_rate', 'bed_tier_Micro', 'domains_reporting', 'total_costs', 'beds', 'hospital_type_Childrens', 'revenue_tier_Q3_Mid-High', 'mort_better', 'bed_tier_Small', 'revenue_tier_Q4_High', 'total_liabilities', 'salary_cost_share', 'ptexp_participation', 'birthing_center', 'revenue_tier_Q2_Mid-Low', 'overhead_cost_share', 'hospital_ownership_Government', 'cbsa', 'readm_better', 'medicaid_net_revenue', 'emergency_services', 'operating_margin', 'hospital_type_Long-term', 'total_assets', 'depreciation_cost_share', 'bed_tier_Major', 'contract_labor_share', 'charity_care_pct_revenue', 'liability_ratio', 'medicaid_revenue_share', 'hospital_ownership_Non-Profit', 'hosp

## Clean Final Joined Data


In [21]:
#drop low importance features
df_app = df_app.drop(columns=[col for col in low_impact_features if col in df_app.columns])
df_app.columns


Index(['ccn', 'report_year', 'y_true', 'LogisticRegression_proba',
       'LogisticRegression_pred', 'RandomForest_proba', 'RandomForest_pred',
       'XGBoost_proba', 'XGBoost_pred', 'SV_proba', 'SVM_pred',
       'fte_employees', 'cost_to_charge_ratio', 'outpatient_revenue_share',
       'inpatient_revenue_share', 'current_ratio', 'fte_employees_yoy_change',
       'bed_tier', 'revenue_tier', 'hospital_type', 'hospital_ownership',
       'hospital_rating', 'engagement_index', 'performance_ratio',
       'total_individuals_affected', 'had_ransomware', 'State'],
      dtype='object')

In [22]:
#drop columns
df_app = df_app.drop(columns=['total_individuals_affected', 'had_ransomware', 'report_year'], errors='ignore')

print(f"Shape: {df_app.shape}")
print(f"\nColumns:\n{df_app.columns.tolist()}")
print(f"\nMissing values:\n{df_app.isnull().sum()[df_app.isnull().sum() > 0]}")
print(f"\nSample:\n{df_app.head(3).to_string()}")

Shape: (5022, 24)

Columns:
['ccn', 'y_true', 'LogisticRegression_proba', 'LogisticRegression_pred', 'RandomForest_proba', 'RandomForest_pred', 'XGBoost_proba', 'XGBoost_pred', 'SV_proba', 'SVM_pred', 'fte_employees', 'cost_to_charge_ratio', 'outpatient_revenue_share', 'inpatient_revenue_share', 'current_ratio', 'fte_employees_yoy_change', 'bed_tier', 'revenue_tier', 'hospital_type', 'hospital_ownership', 'hospital_rating', 'engagement_index', 'performance_ratio', 'State']

Missing values:
fte_employees                 43
cost_to_charge_ratio         653
outpatient_revenue_share     306
inpatient_revenue_share      151
current_ratio                242
fte_employees_yoy_change     110
bed_tier                      26
revenue_tier                 150
hospital_rating             2276
performance_ratio            878
dtype: int64

Sample:
     ccn  y_true  LogisticRegression_proba  LogisticRegression_pred  RandomForest_proba  RandomForest_pred  XGBoost_proba  XGBoost_pred  SV_proba  SVM_pr

## Save Data to CSV & Pickle

In [23]:
from google.colab import drive

drive.mount('/content/drive')
drive_path = '/content/drive/MyDrive/ADS 599 Capstone Project/Data/'
import os
if not os.path.exists(drive_path):
    os.makedirs(drive_path)
df_app.to_csv(os.path.join(drive_path, 'streamlit.csv'), index=False)
print(f"App data saved to: {drive_path}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
App data saved to: /content/drive/MyDrive/ADS 599 Capstone Project/Data/


In [24]:
import pickle

with open("/content/hospital_data.pkl", "wb") as f:
    pickle.dump(df_app, f)

print("Saved!", df_app.shape)

Saved! (5022, 24)


# Streamlit App

## Steamlit App Instructions

# Streamlit App — Setup Instructions

This app requires a few one-time setup steps before it can be launched.
Follow the steps below in order.

---

### Step 1 — Create a Free ngrok Account
ngrok is the service that makes the app accessible in your browser from Google Colab.

1. Go to **https://ngrok.com** and create a free account
2. After signing in, go to **Your Authtoken** on the ngrok dashboard
3. Copy your authtoken — it looks like `2abc123xyz...`

---

### Step 2 — Save Your ngrok Token as a Colab Secret
Colab Secrets let you store your token securely without it appearing in the notebook.

1. In Google Colab, click the **🔑 key icon** in the left sidebar
2. Click **+ Add new secret**
3. Set the name to exactly: `ngrok`
4. Paste your authtoken as the value
5. Toggle **"Notebook access"** to ON

> ⚠️ Do not share your authtoken or paste it directly into the notebook.
> Anyone with your token can use your ngrok account.

---

### Step 3 — Run the Notebook in Order
Once your secret is saved, run all cells from top to bottom.
The key cells for the app are:

| Cell | What it does |
|---|---|
| Install cell | Installs `streamlit` and `pyngrok` |
| Version pin cell | Pins Streamlit to version 1.32.0 for compatibility |
| Pickle cell | Saves the prepared dataset for the app to read |
| `%%writefile` cell | Writes the app code to a file Streamlit can run |
| Launch cell | Starts Streamlit and opens the ngrok tunnel |

---

### Step 4 — Open the App
When the launch cell finishes, it will print:
```
✅ App is live at: NgrokTunnel: "https://xxxx.ngrok-free.app"
```

Click that URL. Your app will open in a new browser tab.
No password required.

---

### Troubleshooting

| Problem | Fix |
|---|---|
| `KeyError: 'ngrok'` | Your Colab secret is not named exactly `ngrok` — check Step 2 |
| `Module not found: pyngrok` | Re-run the install cell |
| Blank page or loading forever | Wait 10 seconds and hard reload (`Ctrl+Shift+R` / `Cmd+Shift+R`) |
| App URL not printed | Increase `time.sleep(8)` to `time.sleep(12)` in the launch cell and rerun |
| `Streamlit running: False` | Re-run the `%%writefile` cell then the launch cell |

In [25]:
#install version of streamlist needed
!pip install streamlit==1.32.0

In [26]:
%%writefile hospital_risk_explorer.py

import streamlit as st
import pandas as pd
import numpy as np

st.set_page_config(
    page_title="Hospital Risk Explorer",
    page_icon="🏥",
    layout="wide",
    initial_sidebar_state="expanded",
)

@st.cache_data
def load_data() -> pd.DataFrame:
    df = pd.read_pickle("/content/hospital_data.pkl")
    return df

df_raw = load_data()

MODEL_PROBA_COLS = {
    "Logistic Regression": "LogisticRegression_proba",
    "Random Forest":       "RandomForest_proba",
    "XGBoost":             "XGBoost_proba",
    "SVM":                 "SV_proba",
}

with st.sidebar:
    st.title("⚙️ Controls")

    selected_model = st.selectbox(
        "🤖 Model",
        options=list(MODEL_PROBA_COLS.keys()),
        index=2,
        help="Choose which ML model's probability score to use for flagging.",
    )
    proba_col = MODEL_PROBA_COLS[selected_model]

    threshold = st.slider(
        "🎯 Risk Threshold",
        min_value=0.01, max_value=0.99, value=0.30, step=0.01,
        help="Hospitals with a model probability ≥ this value are flagged.",
    )

    st.divider()

    all_states = sorted(df_raw["State"].dropna().unique())
    selected_states = st.multiselect(
        "🗺️ States",
        options=all_states,
        default=[],
        placeholder="All states (default)",
    )

    all_types = sorted(df_raw["hospital_type"].dropna().unique())
    selected_types = st.multiselect(
        "🏥 Hospital Type",
        options=all_types,
        default=[],
        placeholder="All types (default)",
    )

    st.divider()
    st.caption("Hospital Risk Explorer | Graduate Capstone Project")

df = df_raw.copy()

if selected_states:
    df = df[df["State"].isin(selected_states)]

if selected_types:
    df = df[df["hospital_type"].isin(selected_types)]

df["flagged"] = (df[proba_col] >= threshold).astype(int)

y_true = df["y_true"]
y_pred = df["flagged"]

n_flagged = int(y_pred.sum())
n_total   = len(df)

TP = int(((y_pred == 1) & (y_true == 1)).sum())
FN = int(((y_pred == 0) & (y_true == 1)).sum())
FP = int(((y_pred == 1) & (y_true == 0)).sum())

recall    = TP / (TP + FN) if (TP + FN) > 0 else 0.0
precision = TP / (TP + FP) if (TP + FP) > 0 else 0.0

st.title("🏥 Hospital Risk Explorer")
st.caption("*Which hospitals should we prioritize, and at what threshold?*")

pct_flagged = n_flagged / n_total * 100 if n_total > 0 else 0
consequence_color = "#d62728" if recall < 0.60 else ("#ff7f0e" if recall < 0.80 else "#2ca02c")

st.markdown(
    f"""
    <div style="background:#eef4ff; border-left:5px solid {consequence_color};
            padding:14px 20px; border-radius:6px; margin-bottom:18px; color:#000000;">
        <b>At a threshold of {threshold:.2f} using {selected_model}:</b>
        &nbsp; {n_flagged} hospitals ({pct_flagged:.1f}% of the filtered set) are flagged for review.<br>
        Of the <b>{TP + FN} truly high-risk hospitals</b> in this view,
        the model <b>catches {TP}</b> ({recall*100:.1f}% recall)
        and <b>misses {FN}</b>.
        There are <b>{FP} false alarms</b>.
    </div>
    """,
    unsafe_allow_html=True,
)

c1, c2, c3, c4, c5, c6 = st.columns(6)

c1.metric("🚩 Flagged",      n_flagged)
c2.metric("✅ Caught (TP)",  TP)
c3.metric("❌ Missed (FN)",  FN)
c4.metric("⚠️ False Alarms", FP)
c5.metric("📡 Recall",       f"{recall*100:.1f}%")
c6.metric("🎯 Precision",    f"{precision*100:.1f}%")

st.divider()

def assign_tier(prob: float, thresh: float) -> str:
    if prob >= thresh + 0.30:
        return "Critical"
    elif prob >= thresh + 0.15:
        return "High"
    elif prob >= thresh:
        return "Medium"
    else:
        return "Low"

df_flagged = df[df["flagged"] == 1].copy()
df_flagged["Risk Tier"] = df_flagged[proba_col].apply(
    lambda p: assign_tier(p, threshold)
)

display_cols = {
    "ccn":                      "CCN",
    "State":                    "State",
    "hospital_type":            "Hospital Type",
    "hospital_ownership":       "Ownership",
    "bed_tier":                 "Bed Tier",
    "hospital_rating":          "Rating",
    "Risk Tier":                "Risk Tier",
    "LogisticRegression_proba": "LR Score",
    "RandomForest_proba":       "RF Score",
    "XGBoost_proba":            "XGB Score",
    "SV_proba":                 "SVM Score",
    "current_ratio":            "Current Ratio",
    "fte_employees":            "FTE Employees",
    "engagement_index":         "Engagement Index",
    "performance_ratio":        "Performance Ratio",
    "inpatient_revenue_share":  "Inpatient Rev Share",
    "outpatient_revenue_share": "Outpatient Rev Share",
    "y_true":                   "Actual High-Risk",
}

available_cols = {k: v for k, v in display_cols.items() if k in df_flagged.columns}

df_display = (
    df_flagged[list(available_cols.keys())]
    .rename(columns=available_cols)
    .reset_index(drop=True)
)

tier_order = {"Critical": 0, "High": 1, "Medium": 2, "Low": 3}
df_display["_sort"] = df_display["Risk Tier"].map(tier_order)
df_display = df_display.sort_values("_sort").drop(columns="_sort").reset_index(drop=True)

st.subheader(f"🚩 Flagged Hospitals — {len(df_display)} records")

if df_display.empty:
    st.info("No hospitals meet the current filter + threshold combination. Try lowering the threshold or broadening the filters.")
else:
    def color_tier(val):
        colors = {
            "Critical": "background-color:#ffd5d5; color:#7a0000; font-weight:bold",
            "High":     "background-color:#ffe8cc; color:#7a3800; font-weight:bold",
            "Medium":   "background-color:#fffacc; color:#5a4a00; font-weight:bold",
            "Low":      "background-color:#d5f5d5; color:#005a00; font-weight:bold",
        }
        return colors.get(val, "")

    format_dict = {
        "LR Score":             "{:.3f}",
        "RF Score":             "{:.3f}",
        "XGB Score":            "{:.3f}",
        "SVM Score":            "{:.3f}",
        "Current Ratio":        "{:.2f}",
        "Inpatient Rev Share":  "{:.2%}",
        "Outpatient Rev Share": "{:.2%}",
    }
    active_format = {k: v for k, v in format_dict.items() if k in df_display.columns}

    styled = (
        df_display.style
        .map(color_tier, subset=["Risk Tier"])
        .format(active_format)
    )

    st.dataframe(styled, use_container_width=True, height=420)

    csv_bytes = df_display.to_csv(index=False).encode("utf-8")
    st.download_button(
        label="⬇️ Download Flagged Hospitals CSV",
        data=csv_bytes,
        file_name=f"flagged_hospitals_{selected_model.replace(' ', '_')}_t{threshold:.2f}.csv",
        mime="text/csv",
        type="primary",
    )

st.divider()

Overwriting hospital_risk_explorer.py


In [27]:
!npm install -g localtunnel

⠙⠹⠸⠼⠴⠦⠧⠇⠏
changed 22 packages in 1s
⠏
⠋3 packages are looking for funding
⠋  run `npm fund` for details
⠋

In [30]:
import subprocess, time, os
from google.colab import userdata
from pyngrok import ngrok

# Kill any existing processes
os.system("pkill -f streamlit")
os.system("pkill -f ngrok")
time.sleep(2)

# Authenticate ngrok from your Colab secret
ngrok.set_auth_token(userdata.get("ngrok"))

# Start Streamlit
streamlit_process = subprocess.Popen(
    ["streamlit", "run", "/content/hospital_risk_explorer.py",
     "--server.port=8501",
     "--server.headless=true",
     "--server.enableCORS=false",
     "--server.enableXsrfProtection=false"],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE,
    text=True
)

# Wait for Streamlit to fully start
time.sleep(8)

# Check Streamlit is running
print("Streamlit running:", streamlit_process.poll() is None)

# Open ngrok tunnel
public_url = ngrok.connect(8501)
print("✅ App is live at:", public_url)

Streamlit running: True
✅ App is live at: NgrokTunnel: "https://abstinently-noneloquent-danyelle.ngrok-free.dev" -> "http://localhost:8501"
